<a href="https://colab.research.google.com/github/Shaxzod1991/ABC_Analysis/blob/main/PL_analyze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import glob
import os
!pip install duckdb
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ***Пути к входным и выходным данным***



In [3]:
path_cost_of_sale = r'/content/sample_data/91_счет'                             #Путь к файлам себестоимости
path_revenue = r'/content/sample_data/90_счет'                                  #Путь к файлам Выручки
path_overheads = r'/content/sample_data/94_счет'                                #Путь к файлам расходов периода
path_output = r'/content/drive/MyDrive/Проекты_Санег/Коды_только/Результаты'

# ***Функции преобразования и обработки данных***

In [4]:
#______________________________________________Функция для изменения на тип "Дата"______________________________________________
def def_time_type(df, col):
    df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', dayfirst=True, errors = 'coerce')
    return df

#______________________________________________Функция для изменения типов чисел________________________________________________
def def_num_type(df, col):
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace(r'\s+', '', regex = True)
    df[col] = df[col].str.replace(',', '.', regex = True)
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

#______________________________Функция для объединения файлов выгрузок из 1С в формате txt._____________________________________
def def_load_and_combine(path_name, search_key = 'Период'):
    file_paths = glob.glob(os.path.join(path_name, '*txt'))
    all_frame = []

    for f_path in file_paths:

      line_skip_row = 0
      with open(f_path, 'r', encoding = 'utf - 8') as f_obj:
        for numb, row_name in enumerate(f_obj):
          if search_key in row_name:
            line_skip_row = numb
            break

      df = pd.read_csv(f_path, encoding = 'utf-8', skiprows=line_skip_row, sep = '\t')
      all_frame.append(df)

    if all_frame:                                                        # Объединяем всё, если файлы были найдены
          combined_df = pd.concat(all_frame, ignore_index=True)
          return combined_df
    else:
          print(f"Предупреждение: В папке '{path_name}' файлы .txt не найдены.")
          return pd.DataFrame()                                          # Возвращаем пустой DataFrame, чтобы код не падал дальше

#__________________________________________Функция для переименовании названий столбцов ___________________________________________
def def_col_rename(df):

  columns_name = {'Период': 'Дата',
                'Документ': 'Документ',
                'Аналитика Дт': 'Аналитика_Дт',
                'Аналитика Кт': 'Аналитика_Кт',
                'Дебет': 'Счет_Дт',
                'Кредит': 'Счет_Кт',
                'Unnamed: 6': 'Сумма_Дт',
                'Unnamed: 9': 'Сумма_Кт'}

  df = df.rename(columns=columns_name)
  return df

# ***Себестоимость (Cost of Sales). Объединение и преобразование выгрузок***

In [5]:
#______________________________Применение функции к файлам себестоимости _____________________________________________________
cdf_cost_of_sale = def_load_and_combine(path_cost_of_sale, search_key = 'Период')
cdf_cost_of_sale = def_col_rename(cdf_cost_of_sale)
cdf_cost_of_sale = def_time_type(cdf_cost_of_sale, 'Дата')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Дт')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Кт')

,Дата,Документ,Аналитика_Дт,Аналитика_Кт,Показатель,Счет_Дт,Сумма_Дт,Unnamed: 7,Счет_Кт,Сумма_Кт,Unnamed: 10
0,NaT,NaN,NaN,NaN,NaN,Счет,NaN,NaN,Счет,NaN,NaN
1,2026-02-18,Реализация товаров и услуг 00000000516 от 18.0...,<...>\r\nГазовый конденсат,<...>\r\nКонденсат Лукойл\r\nСклад готовой про...,БУ,9120.11,"2,282,892,623.66",NaN,2111,"2,282,892,623.66",NaN
2,NaT,NaN,NaN,NaN,Кол.,NaN,NaN,NaN,NaN,590.97,NaN
3,2026-02-20,Реализация товаров и услуг 00000000517 от 20.0...,<...>\r\nГазовый конденсат,<...>\r\nКонденсат Лукойл\r\nСклад готовой про...,БУ,9120.11,"2,084,793,388.12",NaN,2111,"2,084,793,388.12",NaN
4,NaT,NaN,NaN,NaN,Кол.,NaN,NaN,NaN,NaN,539.69,NaN
2763,NaT,NaN,NaN,NaN,БУ,"724 903 457 192,14",NaN,NaN,"724 903 457 192,14",NaN,NaN
2764,NaT,NaN,NaN,NaN,Кол.,NaN,NaN,NaN,NaN,NaN,NaN
2765,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2766,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2767,NaT,(должность),NaN,(подпись),NaN,(расшифровка подписи),NaN,NaN,NaN,NaN,NaN


## ***Обработка себестоимости c использованием DuckDB SQL***

In [6]:
#__________________________________________________Обработка в SQL ____________________________________________________________

# 1. Удаляем старую таблицу, если она осталась, и создаем заново
duckdb.execute("DROP TABLE IF EXISTS cost_of_sale")

create_empty_table = """
CREATE TEMPORARY TABLE cost_of_sale (
    "Дата" DATE,
    "Номер_документа" VARCHAR,
    "Аналитика" VARCHAR,
    "Счет_Дт" VARCHAR,
    "Счет_Кт" VARCHAR,
    "Себес_Сумма" DECIMAL(20,2),
    "Кол-во" DECIMAL(10,5)
);
"""
duckdb.execute(create_empty_table)

insert_query = """
INSERT INTO cost_of_sale
   WITH t1 AS (
    SELECT
        *,
        CASE
            WHEN "Показатель" = 'БУ' AND LEFT("Счет_Дт", 2) = '91' THEN lead("Сумма_Кт") OVER()
            ELSE (lead("Сумма_Кт") OVER())*-1
        END AS "Кол-во"
    FROM cdf_cost_of_sale
    WHERE "Показатель" IN ('БУ', 'Кол.')
)
SELECT
    "Дата",
    STRING_SPLIT("Документ", CHR(10))[1] AS "Номер_документа",
    CASE
       WHEN LEFT("Счет_Дт", 2) = '91' AND "Аналитика_Дт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[2]
        WHEN LEFT("Счет_Кт", 2) = '91' AND "Аналитика_Кт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[2]
    END AS "Аналитика",
    "Счет_Дт",
    "Счет_Кт",
    CASE
        WHEN LEFT("Счет_Дт", 2) = '91' THEN "Сумма_Дт"
        WHEN LEFT("Счет_Кт", 2) = '91' THEN "Сумма_Кт" * -1
        ELSE 0
    END AS "Себес_Сумма",
    "Кол-во"
FROM t1
WHERE "Дата" IS NOT NULL AND "Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%'
ORDER BY "Дата"
"""
duckdb.execute(insert_query)

# 2. Выполняем запрос через DuckDB и сразу превращаем результат в новый датафрейм
cost_of_sale = duckdb.query("SELECT * FROM cost_of_sale").df()

pd.concat([cost_of_sale.head(5), cost_of_sale.tail(5)])

,Дата,Номер_документа,Аналитика,Счет_Дт,Счет_Кт,Себес_Сумма,Кол-во
0,2026-01-01,Реализация товаров и услуг 00000000047 от 01.0...,"Дизельное топливо ТД 0,5",9110.51,2810.2,"455,039,863.50",57.90
1,2026-01-01,Реализация товаров и услуг 00000000048 от 01.0...,"Дизельное топливо ТД 0,5",9110.51,2810.2,"455,354,226.10",57.94
2,2026-01-01,Реализация товаров и услуг 00000000049 от 01.0...,"Дизельное топливо ТД 0,5",9110.51,2810.2,"455,354,226.10",57.94
3,2026-01-01,Реализация товаров и услуг 00000000050 от 01.0...,"Дизельное топливо ТД 0,5",9110.51,2810.2,"460,069,665.10",58.54
4,2026-01-01,Реализация товаров и услуг 00000000051 от 01.0...,"Дизельное топливо ТД 0,5",9110.51,2810.2,"460,069,665.10",58.54
1319,2026-03-31,Операция (бухгалтерский учет) 00000001741 от 3...,Возмещение затрат (другие),9130.90,2310.5,"13,294,981.48",NaN
1320,2026-03-31,Операция (бухгалтерский учет) 00000001741 от 3...,Возмещение затрат (другие),9130.90,2310.5,"112,151,534.38",NaN
1321,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"3,064,202.72",NaN
1322,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"3,716,277.32",NaN
1323,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"30,968,977.68",NaN


# ***Выручка (Revenue). Объединение и преобразование выгрузок***

In [7]:
#_________________________________________Применение функции к файлам себестоимости ___________________________________________
cdf_revenue = def_load_and_combine(path_revenue, search_key = 'Период')
cdf_revenue = def_col_rename(cdf_revenue)
cdf_revenue = def_time_type(cdf_revenue, 'Дата')
cdf_revenue = cdf_revenue.rename(columns={'Unnamed: 5': 'Сумма_Дт', 'Unnamed: 8': 'Сумма_Кт', 'Сумма_Дт': 'Удалить_1', 'Сумма_Кт': 'Удалить_2'})
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Дт')
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Кт')

,Дата,Документ,Аналитика_Дт,Аналитика_Кт,Счет_Дт,Сумма_Дт,Удалить_1,Счет_Кт,Сумма_Кт,Удалить_2,Текущее сальдо,Unnamed: 11
0,NaT,NaN,NaN,NaN,Счет,NaN,NaN,Счет,NaN,NaN,NaN,NaN
1,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"0,00"
2,2026-01-01,Реализация товаров и услуг 00000000047 от 01.0...,"""Temiryo'l yonilg'i ta'minot"" AJ\r\nДоговор №4...","Дизельное топливо ТД 0,5\r\n12%",4010.5,NaN,NaN,9010.51,"584,972,333.30",NaN,К,"584 972 333,30"
3,2026-01-01,Реализация товаров и услуг 00000000048 от 01.0...,"""Temiryo'l yonilg'i ta'minot"" AJ\r\nДоговор №4...","Дизельное топливо ТД 0,5\r\n12%",4010.5,NaN,NaN,9010.51,"585,376,459.27",NaN,К,"1 170 348 792,57"
4,2026-01-01,Реализация товаров и услуг 00000000049 от 01.0...,"""Temiryo'l yonilg'i ta'minot"" AJ\r\nДоговор №4...","Дизельное топливо ТД 0,5\r\n12%",4010.5,NaN,NaN,9010.51,"585,376,459.27",NaN,К,"1 755 725 251,84"
944,2026-03-31,Регламентная операция 00000000034 от 31.03.202...,<...>,Прибыль (убыток) от продаж,9010.20,"1,762,935,821.00",NaN,9900.1,NaN,NaN,NaN,NaN
945,NaT,NaN,NaN,NaN,"1 183 051 997 233,67",NaN,NaN,"1 183 051 997 233,67",NaN,NaN,NaN,"0,00"
946,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
947,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
948,NaT,(должность),NaN,(подпись),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## ***Обработка выручки с использованием DuckDB SQL***

In [8]:
#__________________________________________________Обработка в SQL ____________________________________________________________

#Удаляем старую таблицу, если она осталась, и создаем заново
duckdb.execute("DROP TABLE IF EXISTS revenue_SQL")

create_empty_table_revenue = """
    CREATE TEMPORARY TABLE revenue_SQL (
        "Дата" DATE,
        "Номер_документа" VARCHAR,
        "Аналитика" VARCHAR,
        "Счет_Дт" VARCHAR,
        "Счет_Кт" VARCHAR,
        "Выручка_Сумма" DECIMAL(20,2)
        )
"""
duckdb.execute(create_empty_table_revenue)

duckdb.register("duck_cdf_revenue", cdf_revenue)

insert_query_revenue = """
    INSERT INTO revenue_SQL
    SELECT
      "Дата",
       CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
      END AS "Номер_документа",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[1]
      END AS "Аналитика",
      "Счет_Дт",
      "Счет_Кт",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN "Сумма_Дт" * -1
        WHEN "Счет_Кт" LIKE '90%' THEN "Сумма_Кт"
      END AS "Выручка_Сумма"
    FROM duck_cdf_revenue
    WHERE ("Счет_Дт" LIKE '90%' OR "Счет_Кт" LIKE '90%') AND ("Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%')
"""
duckdb.execute(insert_query_revenue)

revenue = duckdb.query("select * from revenue_SQL").df()


,Дата,Номер_документа,Аналитика,Счет_Дт,Счет_Кт,Выручка_Сумма
0,2026-01-01,Реализация товаров и услуг 00000000047 от 01.0...,"Дизельное топливо ТД 0,5\r",4010.5,9010.51,"584,972,333.30"
1,2026-01-01,Реализация товаров и услуг 00000000048 от 01.0...,"Дизельное топливо ТД 0,5\r",4010.5,9010.51,"585,376,459.27"
2,2026-01-01,Реализация товаров и услуг 00000000049 от 01.0...,"Дизельное топливо ТД 0,5\r",4010.5,9010.51,"585,376,459.27"
3,2026-01-01,Реализация товаров и услуг 00000000050 от 01.0...,"Дизельное топливо ТД 0,5\r",4010.5,9010.51,"591,438,348.73"
4,2026-01-01,Реализация товаров и услуг 00000000051 от 01.0...,"Дизельное топливо ТД 0,5\r",4010.5,9010.51,"591,438,348.73"
893,2026-03-31,Реализация товаров и услуг 00000000991 от 31.0...,Природный газ,4010.5,9010.34,"665,704,687.50"
894,2026-03-31,Реализация товаров и услуг 00000000992 от 31.0...,Природный газ,4010.5,9010.34,"687,246,428.57"
895,2026-03-31,Реализация товаров и услуг 00000000993 от 31.0...,Природный газ,4010.5,9010.34,"660,027,455.36"
896,2026-03-31,Реализация товаров и услуг 00000000994 от 31.0...,Хранение ГСМ,4120.1,9030.20,"487,527,791.14"
897,2026-03-31,Реализация товаров и услуг 00000000996 от 31.0...,Хранение ГСМ,4120.1,9030.20,"85,034,300.00"


# ***Объединение данных по выручке и себестоимости с использованием DuckDB SQL***

In [9]:
cost_of_sale['Аналитика'] = cost_of_sale['Аналитика'].str.strip()
revenue['Аналитика'] = revenue['Аналитика'].str.strip()

#__________________________________________________Обработка в SQL ____________________________________________________________
duckdb.execute("DROP TABLE IF EXISTS group_rev")

group_rev = """
    CREATE TEMPORARY TABLE group_rev AS
    SELECT
      "Дата",
      "Аналитика",
      SUM("Выручка_Сумма") AS "Выручка_Сумма"
    FROM revenue
    GROUP BY
      "Дата",
      "Аналитика"
"""
duckdb.execute(group_rev)

pivot_query = """
    with t1 as (select
      "Дата",
      "Аналитика",
      SUM("Себес_Сумма") AS "Себес_Сумма",
      SUM("Кол-во") AS "Кол-во"
    from cost_of_sale
    group by
      "Дата",
      "Аналитика"),
       t2 as (select
        *
      from t1
      full join group_rev on t1."Дата" = group_rev."Дата" and t1."Аналитика" = group_rev."Аналитика")
        select
          CASE
            when "Дата" is null then "Дата_1"
            else "Дата"
          END AS "Дата",
            CASE
                when "Аналитика" is null then "Аналитика_1"
                else "Аналитика"
            END as "Аналитика",
          "Выручка_Сумма",
          "Себес_Сумма",
          "Кол-во"
        from t2
        order by "Дата", "Выручка_Сумма"
        """
reven_cost_of_sale = duckdb.query(pivot_query).df()


,Дата,Аналитика,Выручка_Сумма,Себес_Сумма,Кол-во
0,2026-01-01,"Дизельное топливо ТД 0,5","7,615,753,796.96","5,924,163,197.00",753.80
1,2026-01-02,"Дизельное топливо ТД 0,5","4,085,309,372.98","3,177,891,523.40",404.36
2,2026-01-02,Дизельное топливо З-2 (Зим),"5,718,494,925.00","3,008,098,283.23",526.02
3,2026-01-04,Дизельное топливо З-2 (Зим),"618,356,700.00","320,438,953.89",56.88
4,2026-01-04,Авиационное топливо JET A-1,"677,410,714.29","327,766,214.14",56.20
263,2026-03-31,Скважина №1018 Газли,NaN,18.50,NaN
264,2026-03-31,Скважина №390 Газли,NaN,"-457,294.84",NaN
265,2026-03-31,Скважина №1021 Газли,NaN,38.92,NaN
266,2026-03-31,"Фракция легких углеводородов"" Ts 05767930-250:...",NaN,"6,505,486,076.91",NaN
267,2026-03-31,Общие GRDC,NaN,"79,021,829.24",NaN


# ***Расходы периода***

In [10]:
df_overheads = def_load_and_combine(path_overheads, search_key = 'Период')

def_time_type(df_overheads, 'Период')
def_num_type(df_overheads, 'Unnamed: 5')
def_num_type(df_overheads, 'Unnamed: 8')
df_overheads = df_overheads.iloc[:, [0, 1, 2, 3,4,5,7,8]]
df_overheads = df_overheads.rename(columns={'Период': 'Дата','Аналитика Дт': 'Аналитика_Дт', 'Аналитика Кт': 'Аналитика_Кт' , 'Дебет': 'Счет_Дт','Unnamed: 5': 'Сумма_Дт', 'Кредит': 'Счет_Кт' ,'Unnamed: 8': 'Сумма_Кт'})
df_overheads['Сумма_Дт'] = df_overheads['Сумма_Дт'].fillna(0)
df_overheads['Сумма_Кт'] = df_overheads['Сумма_Кт'].fillna(0)

## ***Обработка расходов периода в DuckDB SQL***

In [11]:
query_overh = """

    WITH T1 AS (SELECT
        *,
        CASE
            WHEN "Счет_Дт" LIKE '9410%' OR "Счет_Кт" LIKE '9410%' THEN '9410'
            WHEN "Счет_Дт" LIKE '9420%' OR "Счет_Кт" LIKE '9420%' THEN '9420'
            WHEN "Счет_Дт" LIKE '9430%' OR "Счет_Кт" LIKE '9430%' THEN '9430'
        END AS "Счет_Загрузки",
        CASE
            WHEN "Счет_Дт" LIKE '94%' THEN STRING_SPLIT(REPLACE("Аналитика_Дт", CHR(13), ''), CHR(10))[3]
            WHEN "Счет_Кт" LIKE '94%' THEN STRING_SPLIT(REPLACE("Аналитика_Кт", CHR(13), ''), CHR(10))[3]
        END AS "Аналитика"
        FROM df_overheads
        WHERE
            "Дата" IS NOT NULL AND ("Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%')
        )
            SELECT
                "Дата",
                "Аналитика_Дт",
                "Аналитика_Кт",
                "Счет_Дт",
                "Счет_Кт",
                "Счет_Загрузки",
                CASE
                    WHEN "Счет_Загрузки" LIKE '9410' THEN 'Коммерческие расходы'
                    WHEN "Счет_Загрузки" LIKE '9420' THEN 'Административные расходы'
                    WHEN "Счет_Загрузки" LIKE '9430' THEN 'Прочие расходы'
                END AS "Название_Статьи",
                CASE
                    WHEN "Аналитика" LIKE '<...>' AND "Счет_Кт" LIKE '92%' THEN 'Списание ОС'
                    WHEN "Аналитика" LIKE '<...>' THEN STRING_SPLIT(REPLACE("Документ", CHR(13), ''), CHR(10))[2]
                    ELSE "Аналитика"
                END AS "Аналитика",
                "Сумма_Дт" - "Сумма_Кт" AS "Сумма"
            FROM T1
"""

overheads = duckdb.sql(query_overh).df()

,Дата,Аналитика_Дт,Аналитика_Кт,Счет_Дт,Счет_Кт,Счет_Загрузки,Название_Статьи,Аналитика,Сумма
0,2026-01-01,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие расходы,"""Fargona neftni qayta ishlash zavod "" MCHJ\r\n...",9420.1,6120.1,9420,Административные расходы,Прочие расходы,"24,769,000.00"
1,2026-01-01,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие расходы,"""Fargona neftni qayta ishlash zavod "" MCHJ\r\n...",9420.1,6120.1,9420,Административные расходы,Прочие расходы,"774,560.00"
2,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...",ТЕХ ПД Ташкент\r\n№ 9178263 от 11.09.2019,9410.1,6010.4,9410,Коммерческие расходы,Услуги по жд,"33,104,643.00"
3,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...","""New logistic"" MCHJ\r\n№ 26 от 21.04.2020",9410.1,6010.4,9410,Коммерческие расходы,Таможенные сборы по экспорту,"32,548,000.00"
4,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...","""New logistic"" MCHJ\r\n№ 26 от 21.04.2020",9410.1,6010.4,9410,Коммерческие расходы,Таможенные сборы по экспорту,"412,000.00"
...,...,...,...,...,...,...,...,...,...
6078,2026-03-31,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие ГСМ,Бензин АИ-100\r\nS10024 - А.Р. Тургунов (ЦАУА)...,9430.1,1030,9430,Прочие расходы,Прочие ГСМ,0.01
6079,2026-03-31,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие ГСМ,Бензин АИ 98\r\nS10024 - А.Р. Тургунов (ЦАУА)\...,9430.1,1030,9430,Прочие расходы,Прочие ГСМ,"9,000.76"
6080,2026-03-31,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие ГСМ,Бензин АИ 92\r\nS10024 - А.Р. Тургунов (ЦАУА)\...,9430.1,1030,9430,Прочие расходы,Прочие ГСМ,"-134,229.63"
6081,2026-03-31,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие ГСМ,Дизтопливо\r\nS10024 - А.Р. Тургунов (ЦАУА)\r\...,9430.1,1030,9430,Прочие расходы,Прочие ГСМ,"236,107.33"


## ***Справочник статьи затрат для Расходов периода***

In [12]:
expense_dict = {
    "Прочие расходы": "Прочие операционные и внереализационные расходы",
    "Услуги по жд": "Транспорт и логистика",
    "Таможенные сборы по экспорту": "Таможенные и государственные услуги",
    "Расходы прошлых лет": "Прочие операционные и внереализационные расходы",
    "Штрафы, пени, неустойки и санкций за нарушение условий договоров": "Штрафы, санкции, страхование и судебные расходы",
    "Перемещена амортизация ОС на новый счет": "Амортизация и внеоборотные активы",
    "Прочие расходы (не относящиеся к основной деятельности)": "Прочие операционные и внереализационные расходы",
    "Возмещение за учебу (УХТА)": "Расходы на персонал",
    "Штрафные санкции по ТехПД": "Штрафы, санкции, страхование и судебные расходы",
    "Содержание железнодорожных путей": "Ремонт и содержание имущества",
    "Командировочные расходы": "Командировочные и представительские расходы",
    "Топливо": "Материалы и производственное обеспечение",
    "Ремонт и техническое обслуживание автотранпорта": "Ремонт и содержание имущества",
    "Ремонт прочего оборудования": "Ремонт и содержание имущества",
    "Проезд в командировках": "Командировочные и представительские расходы",
    "государственные услуги": "Таможенные и государственные услуги",
    "Проживание в командировках": "Командировочные и представительские расходы",
    "Услуги по содержанию территорий и помещений": "Ремонт и содержание имущества",
    "09510 Неотложка хисобидан махсус сувдан фойдаланиг ёки сувни истеъмол килиш учун": "Налоги, сборы и обязательные платежи",
    "Консалтинговые услуги": "Консультационные и экспертные услуги",
    "Вахтовая перевозка": "Транспорт и логистика",
    "Прочие финансовые расходы": "Финансовые и банковские расходы",
    "Хранение сырья, материалов, готового продукта, ОС": "Транспорт и логистика",
    "Экспертиза": "Консультационные и экспертные услуги",
    "Реклама": "Прочие операционные и внереализационные расходы",
    "Банковские услуги": "Финансовые и банковские расходы",
    "Комиссионное сбор и клиринговое обслуживание Уз РТСБ": "Финансовые и банковские расходы",
    "Аренда жилых помещений": "Аренда и содержание объектов",
    "Юридические консультационные услуги": "Консультационные и экспертные услуги",
    "Внедрение и сопровождение программного продукта": "ИТ и связь",
    "Техника и мебель": "Материалы и производственное обеспечение",
    "Спец.одежда и СИЗ": "Материалы и производственное обеспечение",
    "Химреагенты": "Материалы и производственное обеспечение",
    "Прочие материалы": "Материалы и производственное обеспечение",
    "Прочие ГСМ": "Материалы и производственное обеспечение",
    "Командировочные расходы (независимо от основной деятельности)": "Командировочные и представительские расходы",
    "Курьерские услуги": "Транспорт и логистика",
    "Аренда офиса": "Аренда и содержание объектов",
    "Аренда автотранспорта": "Аренда и содержание объектов",
    "Консультационные услуги": "Консультационные и экспертные услуги",
    "Услуги по питанию персонала": "Расходы на персонал",
    "Аренда имущества": "Аренда и содержание объектов",
    "Интернет": "ИТ и связь",
    "Мобильная связь": "ИТ и связь",
    "Услуги связи": "ИТ и связь",
    "Комиссионное вознаграждение за реализацию продуктов ТАСКО": "Комиссионное вознаграждение",
    "Услуги автотранспорта (доставка нефтепродуктов)": "Транспорт и логистика",
    "Услуги по таможенному оформлению": "Таможенные и государственные услуги",
    "Налог на пользвание недрами и разбить на нефть, газ, конденсат": "Налоги, сборы и обязательные платежи",
    "Налог на имущество": "Налоги, сборы и обязательные платежи",
    "Налог на землю": "Налоги, сборы и обязательные платежи",
    "Налог на воду": "Налоги, сборы и обязательные платежи",
    "Налог на прибыль нерезидента по имп.усдуг. (не вычитаемая)": "Налоги, сборы и обязательные платежи",
    "Амортизация": "Амортизация и внеоборотные активы",
    "Канцтовары": "Материалы и производственное обеспечение",
    "Медикаменты": "Материалы и производственное обеспечение",
    "Продукты питания SANEG": "Материалы и производственное обеспечение",
    "Услуги спецтехники": "Ремонт и содержание имущества",
    "Прочие запасные части": "Материалы и производственное обеспечение",
    "Материалы для содержания территорий и помещений": "Ремонт и содержание имущества",
    "Производственный инструмент": "Материалы и производственное обеспечение",
    "Услуги автотранспорта прочие": "Транспорт и логистика",
    "Электрические расходные материалы": "Материалы и производственное обеспечение",
    "Резерв по отпускам": "Расходы на персонал",
    "Электроэнергия": "Коммунальные и энергетические расходы",
    "Прочие затраты на персонал": "Расходы на персонал",
    "ЕСП": "Налоги, сборы и обязательные платежи",
    "1.1.1. Материальные расходы": "Материалы и производственное обеспечение",
    "Зарабатная плата персонала": "Расходы на персонал",
    "Компенсации прочие": "Расходы на персонал",
    "Материальная помощь": "Расходы на персонал",
    "Премия к праздникам": "Расходы на персонал",
    "За использование природного газа": "Налоги, сборы и обязательные платежи",
    "2.2.6. Абонентская плата": "Абонентская плата(АЗС и АГНКС)",
    "Корректировка стоимости списания": "Материалы и производственное обеспечение",
    "Амортизация НМА": "Амортизация и внеоборотные активы",
    "Страхование прочие": "Штрафы, санкции, страхование и судебные расходы",
    "Списание ОС": "Списание ОС",
    "Судебные затраты": "Штрафы, санкции, страхование и судебные расходы",
    "Услуги по жд (не вычитаемая)": "Транспорт и логистика",
    "Расходы по отводу земель": "Прочие операционные и внереализационные расходы",
    "Прочие транспортные расходы": "Транспорт и логистика",
    "Представительские расходы": "Командировочные и представительские расходы",
    "Метрологическая сертификация и поверка измерительных приборов": "Консультационные и экспертные услуги",
    "Пеня по налогу": "Налоги, сборы и обязательные платежи",
    "Пенсионное отчисление по статье 12": "Налоги, сборы и обязательные платежи",
    "Акцизный налог": "Налоги, сборы и обязательные платежи",
    "Начислена амортизация": "Амортизация и внеоборотные активы",
    "Транспортировка сырья по трубопроводу": "Транспорт и логистика",
    "Потери товаров, продуктов при перевозке, хранении и реализации": "Транспорт и логистика",
    "Аренда земельного участка": "Аренда и содержание объектов",
    "Прочие командировочные расходы": "Командировочные и представительские расходы",
    "Электромонтажные работы": "Ремонт и содержание имущества",
    "Услуги по контролю входного к-во нефтепродуктов": "Консультационные и экспертные услуги",
    "Запасные части для автотранспорта": "Материалы и производственное обеспечение",
    "Комиссионное вознаграждение за реализацию продуктов SEGNUM": "Комиссионное вознаграждение"
}

## ***Меппинг статьи затрат Расходов периода***

In [13]:
overheads['Аналитика_свод'] = overheads['Аналитика'].map(expense_dict)


,Дата,Аналитика_Дт,Аналитика_Кт,Счет_Дт,Счет_Кт,Счет_Загрузки,Название_Статьи,Аналитика,Сумма,Аналитика_свод
0,2026-01-01,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие расходы,"""Fargona neftni qayta ishlash zavod "" MCHJ\r\n...",9420.1,6120.1,9420,Административные расходы,Прочие расходы,"24,769,000.00",Прочие операционные и внереализационные расходы
1,2026-01-01,ЦАУ Ташкент\r\nОбщие SANEG\r\nПрочие расходы,"""Fargona neftni qayta ishlash zavod "" MCHJ\r\n...",9420.1,6120.1,9420,Административные расходы,Прочие расходы,"774,560.00",Прочие операционные и внереализационные расходы
2,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...",ТЕХ ПД Ташкент\r\n№ 9178263 от 11.09.2019,9410.1,6010.4,9410,Коммерческие расходы,Услуги по жд,"33,104,643.00",Транспорт и логистика
3,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...","""New logistic"" MCHJ\r\n№ 26 от 21.04.2020",9410.1,6010.4,9410,Коммерческие расходы,Таможенные сборы по экспорту,"32,548,000.00",Таможенные и государственные услуги
4,2026-01-01,"ИП ООО ""SANOAT ENERGETIKA GURUHI""\r\nОбщие SAN...","""New logistic"" MCHJ\r\n№ 26 от 21.04.2020",9410.1,6010.4,9410,Коммерческие расходы,Таможенные сборы по экспорту,"412,000.00",Таможенные и государственные услуги


# ***Экспорт и сохранение итоговых результатов***

In [15]:
exl_path = os.path.join(path_output, 'revenue_cost_of_sale.xlsx')

with pd.ExcelWriter(exl_path) as writer:
    reven_cost_of_sale.to_excel(writer, sheet_name='Выр_себес', index=False)
    cost_of_sale.to_excel(writer, sheet_name='Себестоимость', index=False)
    revenue.to_excel(writer, sheet_name='Выручка', index=False)
    overheads.to_excel(writer, sheet_name='Расходы_периода', index=False)